In [26]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff

from scipy.stats import f_oneway
from ucimlrepo import fetch_ucirepo
from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import KBinsDiscretizer

In [2]:
combined_cycle_power_plant = fetch_ucirepo(id=381)

In [3]:
X : pd.DataFrame = combined_cycle_power_plant.data.features
y : pd.DataFrame = combined_cycle_power_plant.data.targets

## Исследование данных

In [4]:
X.shape

(43824, 11)

In [5]:
y.value_counts()

pm2.5
16.0     626
11.0     596
13.0     589
12.0     578
17.0     572
        ... 
483.0      1
519.0      1
551.0      1
542.0      1
580.0      1
Name: count, Length: 581, dtype: int64

In [6]:
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pm2.5   41757 non-null  float64
dtypes: float64(1)
memory usage: 342.5 KB


In [7]:
ff.create_distplot([y["pm2.5"].dropna()], ["pm2.5"])

In [8]:
X.head()

,year,month,day,hour,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
0,2010,1,1,0,-21,-11.0,1021.0,NW,1.79,0,0
1,2010,1,1,1,-21,-12.0,1020.0,NW,4.92,0,0
2,2010,1,1,2,-21,-11.0,1019.0,NW,6.71,0,0
3,2010,1,1,3,-21,-14.0,1019.0,NW,9.84,0,0
4,2010,1,1,4,-20,-12.0,1018.0,NW,12.97,0,0


In [9]:
X.describe()

,year,month,day,hour,DEWP,TEMP,PRES,Iws,Is,Ir
count,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000
mean,2012.000000,6.523549,15.727820,11.500000,1.817246,12.448521,1016.447654,23.889140,0.052734,0.194916
std,1.413842,3.448572,8.799425,6.922266,14.433440,12.198613,10.268698,50.010635,0.760375,1.415867
min,2010.000000,1.000000,1.000000,0.000000,-40.000000,-19.000000,991.000000,0.450000,0.000000,0.000000
25%,2011.000000,4.000000,8.000000,5.750000,-10.000000,2.000000,1008.000000,1.790000,0.000000,0.000000
50%,2012.000000,7.000000,16.000000,11.500000,2.000000,14.000000,1016.000000,5.370000,0.000000,0.000000
75%,2013.000000,10.000000,23.000000,17.250000,15.000000,23.000000,1025.000000,21.910000,0.000000,0.000000
max,2014.000000,12.000000,31.000000,23.000000,28.000000,42.000000,1046.000000,585.600000,27.000000,36.000000


In [10]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    43824 non-null  int64  
 1   month   43824 non-null  int64  
 2   day     43824 non-null  int64  
 3   hour    43824 non-null  int64  
 4   DEWP    43824 non-null  int64  
 5   TEMP    43824 non-null  float64
 6   PRES    43824 non-null  float64
 7   cbwd    43824 non-null  str    
 8   Iws     43824 non-null  float64
 9   Is      43824 non-null  int64  
 10  Ir      43824 non-null  int64  
dtypes: float64(3), int64(7), str(1)
memory usage: 3.7 MB


In [13]:
df = X.copy(deep=True)
df["pm2.5"] = y["pm2.5"]

In [21]:
df = df.dropna()

## Статистические тесты

In [22]:
num_cols = ["DEWP","TEMP","PRES","Iws","Is","Ir"]
px.imshow(df[num_cols + ["pm2.5"]].corr(), text_auto=True)


In [23]:
groups = [group["pm2.5"].values for _, group in df.groupby("cbwd")]
f_stat, p_val = f_oneway(*groups)
print("F:", f_stat, "p-value:", p_val)

F: 844.2473886382414 p-value: 0.0


In [35]:
df["pm2_5_bin"] = (df["pm2.5"] > df["pm2.5"].median()).astype(int)
df["cbwd_cat"] = LabelEncoder().fit_transform(df["cbwd"])

X = df[["cbwd_cat"]]
y = df["pm2_5_bin"]

chi_scores, p_values = chi2(X, y)

print("Chi-square score:", chi_scores[0])
print(f"p-value: {p_values[0]:.5f}")

Chi-square score: 1239.3125616785953
p-value: 0.00000
